# MAML for Regime-Aware Portfolio Allocation 🚀

This notebook trains a meta-learning model (MAML) to adapt to different market regimes.

**Approach:**
- Meta-learn from 74 training tasks across 3 market regimes
- Adapt to new tasks with just 5 gradient steps
- Compare against supervised baseline

**Expected Runtime:** ~15-20 minutes on Colab CPU

## 📦 Setup & Installation

In [ ]:
# Mount Google Drive (if data is stored there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository (or upload manually)
!git clone https://github.com/YOUR_USERNAME/maml-dynamic-portfolio-allocation.git
%cd maml-dynamic-portfolio-allocation

In [ ]:
# Install dependencies
!pip install -q torch pandas matplotlib tqdm scikit-learn

## 🔍 Verify Dataset

In [ ]:
import json
import pandas as pd

# Check if data files exist
!ls data/processed/

# Load task splits
with open('data/processed/task_split_indices_v2_no_leakage.json', 'r') as f:
    splits = json.load(f)

print(f"\n✅ Dataset loaded successfully!")
print(f"Train tasks: {len(splits['task_indices']['train_task_ids'])}")
print(f"Val tasks: {len(splits['task_indices']['val_task_ids'])}")
print(f"Test tasks: {len(splits['task_indices']['test_task_ids'])}")

## 🧠 Train MAML Model

In [ ]:
# Train MAML
!python scripts/train_maml.py \
    --epochs 50 \
    --meta_batch_size 8 \
    --inner_lr 0.01 \
    --outer_lr 0.001 \
    --inner_steps 5 \
    --hidden_size 64 \
    --output_dir experiments/maml_run1

## 📊 View Results

In [ ]:
# Load and display results
import json
from IPython.display import Image, display

with open('experiments/maml_run1/results.json', 'r') as f:
    results = json.load(f)

print("=" * 60)
print("MAML Test Results")
print("=" * 60)
print(f"Overall Test MSE: {results['test_loss']:.6f}")
print("\nPer-Regime Performance:")
print(f"  Regime 0 (Bearish): {results['regime_losses']['0']:.6f}")
print(f"  Regime 1 (Bullish): {results['regime_losses']['1']:.6f}")
print(f"  Regime 2 (Crisis):  {results['regime_losses']['2']:.6f}")

# Display training curves
print("\n📈 Training Curves:")
display(Image('experiments/maml_run1/training_curves.png'))

## 🔄 Train Supervised Baseline

In [ ]:
# Train supervised baseline for comparison
!python scripts/train_baseline.py \
    --epochs 50 \
    --lr 0.001 \
    --batch_size 32 \
    --hidden_size 64 \
    --output_dir experiments/baseline_run1

## 📊 Compare MAML vs Baseline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Load both results
with open('experiments/maml_run1/results.json', 'r') as f:
    maml_results = json.load(f)

with open('experiments/baseline_run1/results.json', 'r') as f:
    baseline_results = json.load(f)

# Prepare data
regimes = ['Regime 0\n(Bearish)', 'Regime 1\n(Bullish)', 'Regime 2\n(Crisis)']
maml_losses = [
    maml_results['regime_losses']['0'],
    maml_results['regime_losses']['1'],
    maml_results['regime_losses']['2']
]
baseline_losses = [
    baseline_results['regime_losses'][0],
    baseline_results['regime_losses'][1],
    baseline_results['regime_losses'][2]
]

# Create comparison plot
x = np.arange(len(regimes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, maml_losses, width, label='MAML', color='#2ecc71')
rects2 = ax.bar(x + width/2, baseline_losses, width, label='Supervised', color='#e74c3c')

ax.set_ylabel('Test MSE', fontsize=12)
ax.set_title('MAML vs Supervised Baseline: Per-Regime Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(regimes)
ax.legend(fontsize=12)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.4f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.savefig('experiments/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print("\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"\nOverall Test MSE:")
print(f"  MAML:       {maml_results['test_loss']:.6f}")
print(f"  Baseline:   {baseline_results['test_loss']:.6f}")
improvement = (baseline_results['test_loss'] - maml_results['test_loss']) / baseline_results['test_loss'] * 100
print(f"  Improvement: {improvement:+.2f}%")

print(f"\nRegime 2 (Crisis) Performance:")
r2_maml = maml_results['regime_losses']['2']
r2_baseline = baseline_results['regime_losses'][2]
r2_improvement = (r2_baseline - r2_maml) / r2_baseline * 100
print(f"  MAML:       {r2_maml:.6f}")
print(f"  Baseline:   {r2_baseline:.6f}")
print(f"  Improvement: {r2_improvement:+.2f}%")

if r2_improvement > 0:
    print("\n✅ MAML shows better crisis adaptation!")
else:
    print("\n⚠️ Baseline performs better on crisis tasks")

## 💾 Save Models to Drive

In [ ]:
# Copy results to Google Drive for persistence
!cp -r experiments/ /content/drive/MyDrive/maml_experiments/
print("✅ Results saved to Google Drive!")

## 🎯 Next Steps

**If MAML shows good crisis adaptation:**
1. ✅ Run Leave-One-Crisis-Out evaluation
2. ✅ Try different hyperparameters (inner_steps, learning rates)
3. ✅ Proceed to portfolio backtesting (Option 2)

**If MAML underperforms:**
1. ⚠️ Try LSTM architecture (temporal dependencies)
2. ⚠️ Add explicit regime conditioning (one-hot)
3. ⚠️ Increase model capacity (hidden_size)